# DeFakeX — Audited Three-Expert Fusion

This notebook rebuilds the fusion experiment with a clean and reproducible protocol.

### Experts
1. **Frequency branch** — spectral / AI-generation evidence  
2. **Original spatial branch** — manipulation-sensitive evidence  
3. **Auxiliary spatial head** — real-domain-robust evidence  

### Improvements over the previous fusion notebook
- No silent replacement of broken images with black tensors.
- OpenFake paths are audited before use.
- HEIC/HEIF phone images are supported.
- FF++ evaluation is a **pure FF++ test**, not a mixed dataset.
- CelebDF remains an untouched cross-dataset test.
- Phone images are split before training and retain a held-out test subset.
- All expert metrics are recomputed on exactly the same samples.
- Logistic-regression and nonlinear MLP fusion baselines are both trained.
- Fusion uses three expert logits plus small quality/conflict features.
- Model and threshold selection penalize real/fake or scenario collapse.

> The notebook selects the strongest validation model; it cannot guarantee a particular score before execution.

In [ ]:
# ============================================================
# CELL 1 — INSTALLS
# ============================================================
!pip -q install pillow-heif

In [ ]:
# ============================================================
# CELL 2 — IMPORTS AND CONFIGURATION
# ============================================================
import os, csv, json, copy, random, hashlib, warnings
from pathlib import Path
from dataclasses import dataclass
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from PIL import Image, ImageFile, UnidentifiedImageError
from pillow_heif import register_heif_opener
register_heif_opener()
ImageFile.LOAD_TRUNCATED_IMAGES = True

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from torchvision.models import efficientnet_b3

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, classification_report
)

from tqdm.auto import tqdm
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PIN_MEMORY = torch.cuda.is_available()
NUM_WORKERS = 2 if torch.cuda.is_available() else 0

TARGET_SIZE = 224
RESIZE_SIZE = 256
FFT_CLIP_MAX = 14.0
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".heic", ".heif"}

# ----------------------- DATASETS ----------------------------
STYLEGAN_ROOT = Path(
    "/kaggle/input/datasets/xhlulu/140k-real-and-fake-faces/"
    "real_vs_fake/real-vs-fake/train"
)
FLICKR_ROOT = Path(
    "/kaggle/input/datasets/adityajn105/flickr30k/Images/flickr30k_images"
)
DEEPDETECT_ROOT = Path(
    "/kaggle/input/datasets/ayushmandatta1/deepdetect-2025/ddata/train"
)
FF_SPLIT_ROOT = Path(
    "/kaggle/input/datasets/gradientvoyager/"
    "faceforensics-c23-extracted-faces-100k/dataset_processed_split"
)
PHONE_ROOT = Path("/kaggle/input/datasets/kashirhanif/phone-images")
CELEB_ROOT = Path(
    "/kaggle/input/datasets/pranabr0y/celebdf-v2image-dataset/Celeb_V2"
)

OPENFAKE_MANIFEST = Path(
    "/kaggle/input/datasets/kashirhanif/frequency-model-checkpoint/of_manifest.csv"
)

# Optional: add actual OpenFake roots here if manifest paths are stale.
OPENFAKE_IMAGE_ROOTS = [
    # Path("/kaggle/input/your-openfake-dataset"),
]

# ----------------------- CHECKPOINTS -------------------------
FREQ_CKPT = Path(
    "/kaggle/input/datasets/kashirhanif/frequency-model-checkpoint/best_model.pth"
)
ORIGINAL_SPATIAL_CKPT = Path(
    "/kaggle/input/datasets/kashirhanif/frequency-model-checkpoint/"
    "best_model_spatial.pth"
)
AUXILIARY_HEAD_CKPT = Path(
    "/kaggle/input/datasets/kashirhanif/frequency-model-checkpoint/"
    "spatial_auxiliary_multitask_head.pth"
)

# ----------------------- CAPS / SPLITS -----------------------
CAPS = {
    "stylegan_real": 30000,
    "stylegan_fake": 30000,
    "flickr_real": 20000,
    "deepdetect_fake": 20000,
    "openfake_per_source": 5000,
}

GENERIC_TRAIN_RATIO = 0.70
GENERIC_VAL_RATIO = 0.15
PHONE_TRAIN_RATIO = 0.60
PHONE_VAL_RATIO = 0.20  # remaining 20% is held-out test

CELEB_REAL_CAP = 10000
CELEB_FAKE_CAP = 20000

EXTRACT_BATCH_SIZE = 48
FUSION_BATCH_SIZE = 512
FUSION_EPOCHS = 40
FUSION_LR = 3e-4
FUSION_WEIGHT_DECAY = 1e-4
FUSION_PATIENCE = 7

CACHE_PATH = Path("/kaggle/working/defakex_three_expert_features.pt")
OUTPUT_DIR = Path("/kaggle/working")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Device:", DEVICE)

In [ ]:
# ============================================================
# CELL 3 — FAST PATH CHECK
# ============================================================
required_paths = {
    "StyleGAN": STYLEGAN_ROOT,
    "Flickr": FLICKR_ROOT,
    "DeepDetect": DEEPDETECT_ROOT,
    "FF++": FF_SPLIT_ROOT,
    "Phone": PHONE_ROOT,
    "CelebDF": CELEB_ROOT,
    "Frequency checkpoint": FREQ_CKPT,
    "Original spatial checkpoint": ORIGINAL_SPATIAL_CKPT,
    "Auxiliary head checkpoint": AUXILIARY_HEAD_CKPT,
}
for name, path in required_paths.items():
    print(f"{name:<30} {'OK' if path.exists() else 'MISSING'}  {path}")

missing = [name for name, path in required_paths.items() if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Update the configuration paths before continuing. Missing: "
        + ", ".join(missing)
    )

In [ ]:
# ============================================================
# CELL 4 — REFERENCES AND SAFE IMAGE HELPERS
# ============================================================
@dataclass(frozen=True)
class SampleRef:
    path: str
    label: int       # 0 real, 1 fake
    source: str
    scenario: str    # clean_real, phone_real, ai_fake, deepfake_fake
    split_group: str = ""

def list_images(root: Path, recursive=True):
    if not root.exists():
        return []
    iterator = root.rglob("*") if recursive else root.iterdir()
    return sorted([
        p for p in iterator
        if p.is_file() and p.suffix.lower() in IMG_EXTS
    ])

def deterministic_shuffle(items, seed=SEED):
    items = list(items)
    random.Random(seed).shuffle(items)
    return items

def cap_paths(paths, cap, seed=SEED):
    paths = deterministic_shuffle(paths, seed)
    return paths if cap is None else paths[:cap]

def validate_image(path: Path):
    try:
        with Image.open(path) as image:
            image.load()
            image.convert("RGB")
        return True, ""
    except Exception as exc:
        return False, str(exc)

def stable_group_key(path: Path):
    # Keeps likely video/frame siblings together when names share prefixes.
    stem = path.stem
    tokens = stem.replace("-", "_").split("_")
    return "_".join(tokens[:2]) if len(tokens) >= 2 else stem

def split_paths_grouped(paths, label, source, scenario,
                        train_ratio=GENERIC_TRAIN_RATIO,
                        val_ratio=GENERIC_VAL_RATIO, seed=SEED):
    groups = defaultdict(list)
    for p in paths:
        groups[stable_group_key(p)].append(p)

    group_names = list(groups)
    random.Random(seed).shuffle(group_names)
    n = len(group_names)
    n_train = int(n * train_ratio)
    n_val = int(n * val_ratio)

    split_groups = {
        "train": group_names[:n_train],
        "val": group_names[n_train:n_train+n_val],
        "test": group_names[n_train+n_val:],
    }

    result = {}
    for split, names in split_groups.items():
        result[split] = [
            SampleRef(str(p), label, source, scenario, group)
            for group in names for p in groups[group]
        ]
    return result

In [ ]:
# ============================================================
# CELL 5 — MODEL DEFINITIONS AND ROBUST CHECKPOINT LOADING
# ============================================================
def build_binary_efficientnet():
    model = efficientnet_b3(weights=None)
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3, inplace=True),
        nn.Linear(in_features, 1),
    )
    return model

def unpack_state_dict(checkpoint):
    if not isinstance(checkpoint, dict):
        return checkpoint
    for key in ("model_state_dict", "state_dict", "model"):
        if key in checkpoint and isinstance(checkpoint[key], dict):
            return checkpoint[key]
    return checkpoint

def clean_state_dict(state):
    cleaned = {}
    for key, value in state.items():
        for prefix in ("module.", "model."):
            if key.startswith(prefix):
                key = key[len(prefix):]
        cleaned[key] = value
    return cleaned

def load_binary_model(path):
    model = build_binary_efficientnet()
    checkpoint = torch.load(path, map_location="cpu")
    state = clean_state_dict(unpack_state_dict(checkpoint))
    model.load_state_dict(state, strict=True)
    model.to(DEVICE).eval()
    for parameter in model.parameters():
        parameter.requires_grad = False
    return model, checkpoint

class SpatialAuxiliaryHead(nn.Module):
    def __init__(self, input_dim, num_aux_classes):
        super().__init__()
        self.shared_trunk = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Dropout(0.35),
            nn.Linear(512, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.20),
        )
        self.binary_head = nn.Linear(128, 1)
        self.auxiliary_head = nn.Linear(128, num_aux_classes)

    def forward(self, features):
        shared = self.shared_trunk(features)
        return {
            "binary_logit": self.binary_head(shared).squeeze(1),
            "auxiliary_logits": self.auxiliary_head(shared),
        }

freq_model, freq_checkpoint = load_binary_model(FREQ_CKPT)
original_spatial_model, spatial_checkpoint = load_binary_model(ORIGINAL_SPATIAL_CKPT)

aux_checkpoint = torch.load(AUXILIARY_HEAD_CKPT, map_location="cpu")
aux_embedding_dim = int(aux_checkpoint.get("embedding_dim", 1536))
aux_class_names = aux_checkpoint.get("aux_class_names", ["real", "fake"])
aux_num_classes = int(aux_checkpoint.get("num_aux_classes", len(aux_class_names)))

aux_model = SpatialAuxiliaryHead(aux_embedding_dim, aux_num_classes)
aux_state = clean_state_dict(
    aux_checkpoint.get("model_state_dict", aux_checkpoint.get("state_dict", {}))
)
aux_model.load_state_dict(aux_state, strict=True)
aux_model.to(DEVICE).eval()
for parameter in aux_model.parameters():
    parameter.requires_grad = False

aux_feature_mean = torch.as_tensor(aux_checkpoint["feature_mean"]).float()
aux_feature_std = torch.as_tensor(aux_checkpoint["feature_std"]).float().clamp_min(1e-6)

# The auxiliary head uses embeddings from the original spatial backbone.
spatial_backbone = copy.deepcopy(original_spatial_model)
spatial_backbone.classifier = nn.Identity()
spatial_backbone.to(DEVICE).eval()
for parameter in spatial_backbone.parameters():
    parameter.requires_grad = False

print("Three experts loaded successfully.")

In [ ]:
# ============================================================
# CELL 6 — EXACT BRANCH PREPROCESSING
# ============================================================
spatial_transform = transforms.Compose([
    transforms.Resize((RESIZE_SIZE, RESIZE_SIZE)),
    transforms.CenterCrop(TARGET_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

def preprocess_spatial(image):
    return spatial_transform(image.convert("RGB"))

def preprocess_frequency(image):
    image = image.convert("RGB").resize(
        (RESIZE_SIZE, RESIZE_SIZE), Image.Resampling.BICUBIC
    )
    offset = (RESIZE_SIZE - TARGET_SIZE) // 2
    image = image.crop((offset, offset, offset+TARGET_SIZE, offset+TARGET_SIZE))

    array = np.asarray(image, dtype=np.float32) / 255.0
    tensor = torch.from_numpy(array).permute(2, 0, 1)

    channels = []
    for channel in tensor:
        spectrum = torch.fft.fftshift(torch.fft.fft2(channel))
        magnitude = torch.log1p(torch.abs(spectrum))
        magnitude = torch.clamp(magnitude, 0, FFT_CLIP_MAX) / FFT_CLIP_MAX
        channels.append(magnitude)
    return torch.stack(channels)

def quality_features(image):
    # Small, interpretable quality features used only by fusion.
    gray = np.asarray(image.convert("L").resize((224, 224)), dtype=np.float32) / 255.0
    brightness = float(gray.mean())
    contrast = float(gray.std())
    gx = np.diff(gray, axis=1)
    gy = np.diff(gray, axis=0)
    sharpness = float((gx.var() + gy.var()) / 2.0)
    dark_fraction = float((gray < 0.10).mean())
    bright_fraction = float((gray > 0.90).mean())
    return np.array(
        [brightness, contrast, sharpness, dark_fraction, bright_fraction],
        dtype=np.float32,
    )

In [ ]:
# ============================================================
# CELL 7 — AUDIT OPENFAKE MANIFEST (NO SILENT BLACK TENSORS)
# ============================================================
def resolve_manifest_path(raw_path):
    path = Path(raw_path)
    if path.exists():
        return path

    # Try preserving relative suffix under configured OpenFake roots.
    candidates = []
    raw_parts = path.parts
    for root in OPENFAKE_IMAGE_ROOTS:
        candidates.extend([
            root / path.name,
            root.joinpath(*raw_parts[-3:]) if len(raw_parts) >= 3 else root / path.name,
            root.joinpath(*raw_parts[-2:]) if len(raw_parts) >= 2 else root / path.name,
        ])
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return None

openfake_valid_refs = []
openfake_audit_rows = []

if OPENFAKE_MANIFEST.exists():
    with open(OPENFAKE_MANIFEST, newline="") as handle:
        rows = list(csv.DictReader(handle))

    for row in tqdm(rows, desc="Auditing OpenFake manifest"):
        resolved = resolve_manifest_path(row["path"])
        source_name = row.get("source", "unknown")
        label = int(row["label"])
        source = "openfake_real" if label == 0 else f"openfake_{source_name}"

        if resolved is None:
            openfake_audit_rows.append({
                "raw_path": row["path"], "source": source,
                "status": "missing", "error": "path not resolved"
            })
            continue

        ok, error = validate_image(resolved)
        openfake_audit_rows.append({
            "raw_path": row["path"], "resolved_path": str(resolved),
            "source": source, "status": "valid" if ok else "decode_error",
            "error": error,
        })
        if ok:
            scenario = "clean_real" if label == 0 else "ai_fake"
            openfake_valid_refs.append(
                SampleRef(str(resolved), label, source, scenario, stable_group_key(resolved))
            )

openfake_audit_df = pd.DataFrame(openfake_audit_rows)
if len(openfake_audit_df):
    display(openfake_audit_df["status"].value_counts().to_frame("count"))
    openfake_audit_df.to_csv(OUTPUT_DIR / "openfake_path_audit.csv", index=False)

print("Valid OpenFake references:", len(openfake_valid_refs))

In [ ]:
# ============================================================
# CELL 8 — BUILD CLEAN TRAIN / VAL / MIXED TEST REFERENCES
# ============================================================
train_refs, val_refs, mixed_test_refs = [], [], []

def add_generic_source(paths, label, source, scenario, cap=None, seed=SEED):
    paths = cap_paths(paths, cap, seed)
    split = split_paths_grouped(paths, label, source, scenario, seed=seed)
    train_refs.extend(split["train"])
    val_refs.extend(split["val"])
    mixed_test_refs.extend(split["test"])

add_generic_source(
    list_images(STYLEGAN_ROOT / "real"), 0, "stylegan_real", "clean_real",
    CAPS["stylegan_real"], SEED+1
)
add_generic_source(
    list_images(STYLEGAN_ROOT / "fake"), 1, "stylegan_fake", "ai_fake",
    CAPS["stylegan_fake"], SEED+2
)
add_generic_source(
    list_images(FLICKR_ROOT), 0, "flickr_real", "clean_real",
    CAPS["flickr_real"], SEED+3
)
add_generic_source(
    list_images(DEEPDETECT_ROOT / "fake"), 1, "deepdetect_fake", "ai_fake",
    CAPS["deepdetect_fake"], SEED+4
)

# OpenFake: split independently per source to prevent source domination.
for source in sorted({r.source for r in openfake_valid_refs}):
    source_refs = [r for r in openfake_valid_refs if r.source == source]
    source_refs = deterministic_shuffle(source_refs, SEED)
    cap = CAPS["openfake_per_source"]
    source_refs = source_refs[:cap] if cap else source_refs

    groups = defaultdict(list)
    for ref in source_refs:
        groups[ref.split_group].append(ref)
    names = list(groups)
    random.Random(SEED).shuffle(names)
    n = len(names)
    a, b = int(n*GENERIC_TRAIN_RATIO), int(n*(GENERIC_TRAIN_RATIO+GENERIC_VAL_RATIO))
    for split_name, selected in (
        ("train", names[:a]), ("val", names[a:b]), ("test", names[b:])
    ):
        target = train_refs if split_name == "train" else val_refs if split_name == "val" else mixed_test_refs
        target.extend([r for name in selected for r in groups[name]])

# FF++ uses its official split and includes BOTH real and fake.
FF_FAKE_TYPES = [
    "Deepfakes", "Face2Face", "FaceShifter",
    "FaceSwap", "NeuralTextures", "DeepFakeDetection"
]
pure_ff_test_refs = []

for split_name, target in (
    ("train", train_refs), ("val", val_refs), ("test", pure_ff_test_refs)
):
    split_root = FF_SPLIT_ROOT / split_name
    real_dir = next(
        (split_root / name for name in ("Real", "real", "Original", "original")
         if (split_root / name).exists()),
        None,
    )
    if real_dir:
        target.extend([
            SampleRef(str(p), 0, "ff_Real", "clean_real", stable_group_key(p))
            for p in list_images(real_dir)
        ])

    for fake_type in FF_FAKE_TYPES:
        folder = split_root / fake_type
        if folder.exists():
            target.extend([
                SampleRef(str(p), 1, f"ff_{fake_type}", "deepfake_fake",
                          stable_group_key(p))
                for p in list_images(folder)
            ])

# Phone images: audit, then deterministic grouped split.
phone_paths = list_images(PHONE_ROOT)
valid_phone_paths, invalid_phone_rows = [], []
for path in tqdm(phone_paths, desc="Validating phone images"):
    ok, error = validate_image(path)
    if ok:
        valid_phone_paths.append(path)
    else:
        invalid_phone_rows.append({"path": str(path), "error": error})

pd.DataFrame(invalid_phone_rows).to_csv(
    OUTPUT_DIR / "invalid_phone_images.csv", index=False
)

phone_split = split_paths_grouped(
    valid_phone_paths, 0, "phone_real", "phone_real",
    train_ratio=PHONE_TRAIN_RATIO, val_ratio=PHONE_VAL_RATIO, seed=SEED
)
train_refs.extend(phone_split["train"])
val_refs.extend(phone_split["val"])
phone_test_refs = phone_split["test"]

random.Random(SEED).shuffle(train_refs)
random.Random(SEED+1).shuffle(val_refs)
random.Random(SEED+2).shuffle(mixed_test_refs)
random.Random(SEED+3).shuffle(pure_ff_test_refs)
random.Random(SEED+4).shuffle(phone_test_refs)

def summarize_refs(name, refs):
    print(f"\n{name}: {len(refs):,}")
    print(" labels:", Counter(r.label for r in refs))
    print(" scenarios:", Counter(r.scenario for r in refs))
    print(" sources:", len(set(r.source for r in refs)))

for name, refs in (
    ("Fusion train", train_refs), ("Fusion val", val_refs),
    ("Mixed held-out test", mixed_test_refs),
    ("Pure FF++ test", pure_ff_test_refs),
    ("Phone held-out test", phone_test_refs),
):
    summarize_refs(name, refs)

In [ ]:
# ============================================================
# CELL 9 — BUILD UNTOUCHED CELEBDF TEST
# ============================================================
celeb_real, celeb_fake = [], []
for split_name in ("Train", "Val", "Test", "train", "val", "test"):
    split_root = CELEB_ROOT / split_name
    if not split_root.exists():
        continue
    for name in ("real", "Real"):
        folder = split_root / name
        if folder.exists():
            celeb_real.extend(list_images(folder))
    for name in ("fake", "Fake"):
        folder = split_root / name
        if folder.exists():
            celeb_fake.extend(list_images(folder))

celeb_real = cap_paths(celeb_real, CELEB_REAL_CAP, SEED+10)
celeb_fake = cap_paths(celeb_fake, CELEB_FAKE_CAP, SEED+11)

celeb_test_refs = [
    SampleRef(str(p), 0, "celebdf_real", "clean_real", stable_group_key(p))
    for p in celeb_real
] + [
    SampleRef(str(p), 1, "celebdf_fake", "deepfake_fake", stable_group_key(p))
    for p in celeb_fake
]
random.Random(SEED).shuffle(celeb_test_refs)
summarize_refs("CelebDF zero-shot test", celeb_test_refs)

if not celeb_real or not celeb_fake:
    raise RuntimeError("CelebDF mapping found only one class. Inspect CELEB_ROOT.")

In [ ]:
# ============================================================
# CELL 10 — FEATURE EXTRACTION DATASET
# ============================================================
class ThreeExpertDataset(Dataset):
    def __init__(self, refs):
        self.refs = refs

    def __len__(self):
        return len(self.refs)

    def __getitem__(self, index):
        ref = self.refs[index]
        try:
            with Image.open(ref.path) as image:
                image.load()
                image = image.convert("RGB")
                spatial = preprocess_spatial(image)
                frequency = preprocess_frequency(image)
                quality = torch.from_numpy(quality_features(image))
        except Exception as exc:
            raise RuntimeError(f"Failed to decode {ref.path}: {exc}") from exc

        return spatial, frequency, quality, ref.label, ref.source, ref.scenario, ref.path

@torch.inference_mode()
def extract_three_expert_features(refs, description):
    loader = DataLoader(
        ThreeExpertDataset(refs),
        batch_size=EXTRACT_BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
    )

    rows, labels, sources, scenarios, paths = [], [], [], [], []

    for spatial, frequency, quality, label, source, scenario, path in tqdm(
        loader, desc=description
    ):
        spatial = spatial.to(DEVICE, non_blocking=PIN_MEMORY)
        frequency = frequency.to(DEVICE, non_blocking=PIN_MEMORY)

        with torch.autocast(
            device_type="cuda", dtype=torch.float16, enabled=(DEVICE == "cuda")
        ):
            freq_logit = freq_model(frequency).reshape(-1).float()
            original_logit = original_spatial_model(spatial).reshape(-1).float()
            embedding = spatial_backbone(spatial).float()

        embedding_cpu = embedding.cpu()
        standardized = (
            embedding_cpu - aux_feature_mean
        ) / aux_feature_std
        aux_output = aux_model(standardized.to(DEVICE))
        aux_logit = aux_output["binary_logit"].float()

        # Conflict features are deterministic transformations of logits.
        freq_prob = torch.sigmoid(freq_logit)
        original_prob = torch.sigmoid(original_logit)
        aux_prob = torch.sigmoid(aux_logit)

        expert = torch.stack([
            freq_logit, original_logit, aux_logit,
            freq_prob, original_prob, aux_prob,
            torch.abs(freq_prob-original_prob),
            torch.abs(freq_prob-aux_prob),
            torch.abs(original_prob-aux_prob),
            torch.minimum(torch.minimum(freq_prob, original_prob), aux_prob),
            torch.maximum(torch.maximum(freq_prob, original_prob), aux_prob),
        ], dim=1).cpu()

        rows.append(torch.cat([expert, quality.float()], dim=1))
        labels.extend(torch.as_tensor(label).long().tolist())
        sources.extend(list(source))
        scenarios.extend(list(scenario))
        paths.extend(list(path))

    return {
        "X": torch.cat(rows).numpy().astype(np.float32),
        "y": np.asarray(labels, dtype=np.int64),
        "sources": sources,
        "scenarios": scenarios,
        "paths": paths,
    }

In [ ]:
# ============================================================
# CELL 11 — EXTRACT OR LOAD CACHED FEATURES
# ============================================================
if CACHE_PATH.exists():
    cache = torch.load(CACHE_PATH, map_location="cpu", weights_only=False)
    print("Loaded feature cache:", CACHE_PATH)
else:
    cache = {
        "train": extract_three_expert_features(train_refs, "Fusion train"),
        "val": extract_three_expert_features(val_refs, "Fusion validation"),
        "mixed_test": extract_three_expert_features(mixed_test_refs, "Mixed test"),
        "ff_test": extract_three_expert_features(pure_ff_test_refs, "Pure FF++ test"),
        "celeb_test": extract_three_expert_features(celeb_test_refs, "CelebDF test"),
        "phone_test": extract_three_expert_features(phone_test_refs, "Phone test"),
    }
    torch.save(cache, CACHE_PATH)
    print("Saved feature cache:", CACHE_PATH)

for split, data in cache.items():
    print(split, data["X"].shape, Counter(data["y"]), Counter(data["scenarios"]))

In [ ]:
# ============================================================
# CELL 12 — AUDIT DUPLICATES AND SPLIT LEAKAGE
# ============================================================
def path_hash(path):
    return hashlib.sha1(str(Path(path).resolve()).encode()).hexdigest()

split_hashes = {
    split: {path_hash(p) for p in data["paths"]}
    for split, data in cache.items()
}
for left, right in (
    ("train", "val"), ("train", "mixed_test"), ("train", "ff_test"),
    ("train", "celeb_test"), ("train", "phone_test"),
    ("val", "mixed_test"), ("val", "phone_test"),
):
    overlap = split_hashes[left] & split_hashes[right]
    print(f"{left} vs {right}: overlap={len(overlap)}")
    if overlap:
        raise RuntimeError(f"Data leakage detected between {left} and {right}")

In [ ]:
# ============================================================
# CELL 13 — STANDARDIZE FUSION FEATURES
# ============================================================
scaler = StandardScaler()
X_train = scaler.fit_transform(cache["train"]["X"])
X_val = scaler.transform(cache["val"]["X"])

def transformed(split):
    return scaler.transform(cache[split]["X"])

y_train = cache["train"]["y"]
y_val = cache["val"]["y"]

feature_names = [
    "freq_logit", "original_spatial_logit", "aux_spatial_logit",
    "freq_prob", "original_spatial_prob", "aux_spatial_prob",
    "abs_freq_original", "abs_freq_aux", "abs_original_aux",
    "min_expert_prob", "max_expert_prob",
    "brightness", "contrast", "sharpness", "dark_fraction", "bright_fraction",
]
print("Feature count:", len(feature_names))

In [ ]:
# ============================================================
# CELL 14 — METRICS AND VALIDATION OBJECTIVE
# ============================================================
def binary_metrics(y_true, probabilities, threshold):
    predictions = (probabilities >= threshold).astype(np.int64)
    result = {
        "accuracy": accuracy_score(y_true, predictions),
        "macro_f1": f1_score(y_true, predictions, average="macro", zero_division=0),
        "real_f1": f1_score(y_true, predictions, pos_label=0, zero_division=0),
        "fake_f1": f1_score(y_true, predictions, pos_label=1, zero_division=0),
        "real_recall": recall_score(y_true, predictions, pos_label=0, zero_division=0),
        "fake_recall": recall_score(y_true, predictions, pos_label=1, zero_division=0),
        "auc": roc_auc_score(y_true, probabilities) if len(np.unique(y_true)) == 2 else np.nan,
        "predictions": predictions,
    }
    return result

def scenario_recalls(y_true, probabilities, threshold, scenarios):
    predictions = (probabilities >= threshold).astype(np.int64)
    output = {}
    scenarios = np.asarray(scenarios)
    for scenario in sorted(set(scenarios)):
        mask = scenarios == scenario
        true_class = int(round(y_true[mask].mean()))
        output[scenario] = recall_score(
            y_true[mask], predictions[mask], pos_label=true_class, zero_division=0
        )
    return output

def choose_threshold(y_true, probabilities, scenarios):
    rows = []
    for threshold in np.linspace(0.05, 0.95, 361):
        metrics = binary_metrics(y_true, probabilities, threshold)
        recalls = scenario_recalls(y_true, probabilities, threshold, scenarios)
        worst_scenario = min(recalls.values()) if recalls else 0.0
        recall_gap = abs(metrics["real_recall"] - metrics["fake_recall"])
        score = (
            0.45 * metrics["macro_f1"]
            + 0.30 * worst_scenario
            + 0.20 * min(metrics["real_recall"], metrics["fake_recall"])
            - 0.15 * recall_gap
        )
        rows.append({
            "threshold": threshold, "score": score,
            **{k: v for k, v in metrics.items() if k != "predictions"},
            "worst_scenario_recall": worst_scenario,
            **{f"recall_{k}": v for k, v in recalls.items()},
        })
    frame = pd.DataFrame(rows)
    return frame.loc[frame["score"].idxmax()], frame

def print_metrics(title, y, probabilities, threshold, scenarios=None):
    metrics = binary_metrics(y, probabilities, threshold)
    print("\n" + "="*72)
    print(title)
    print("="*72)
    print(f"Threshold   : {threshold:.4f}")
    for key in ("accuracy", "macro_f1", "real_f1", "fake_f1",
                "real_recall", "fake_recall", "auc"):
        print(f"{key:<13}: {metrics[key]:.4f}")
    print("Confusion matrix:")
    print(confusion_matrix(y, metrics["predictions"], labels=[0, 1]))
    if scenarios is not None:
        print("Scenario recalls:", scenario_recalls(y, probabilities, threshold, scenarios))
    return metrics

In [ ]:
# ============================================================
# CELL 15 — LOGISTIC-REGRESSION FUSION BASELINE
# ============================================================
logistic = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    C=0.5,
    random_state=SEED,
)
logistic.fit(X_train, y_train)

log_val_probs = logistic.predict_proba(X_val)[:, 1]
log_threshold_row, log_threshold_table = choose_threshold(
    y_val, log_val_probs, cache["val"]["scenarios"]
)
LOGISTIC_THRESHOLD = float(log_threshold_row["threshold"])
display(log_threshold_row.to_frame().T)

print_metrics(
    "LOGISTIC FUSION — VALIDATION", y_val, log_val_probs,
    LOGISTIC_THRESHOLD, cache["val"]["scenarios"]
)

In [ ]:
# ============================================================
# CELL 16 — SCENARIO-BALANCED MLP DATASET
# ============================================================
class FusionFeatureDataset(Dataset):
    def __init__(self, X, y, scenarios):
        self.X = torch.as_tensor(X).float()
        self.y = torch.as_tensor(y).float()
        self.scenarios = list(scenarios)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, index):
        return self.X[index], self.y[index]

scenario_counts = Counter(cache["train"]["scenarios"])
sample_weights = np.asarray([
    1.0 / scenario_counts[scenario]
    for scenario in cache["train"]["scenarios"]
], dtype=np.float64)

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True,
)

train_loader = DataLoader(
    FusionFeatureDataset(X_train, y_train, cache["train"]["scenarios"]),
    batch_size=FUSION_BATCH_SIZE,
    sampler=sampler,
    num_workers=0,
)
val_loader = DataLoader(
    FusionFeatureDataset(X_val, y_val, cache["val"]["scenarios"]),
    batch_size=FUSION_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)

class FusionMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.Dropout(0.25),
            nn.Linear(64, 24),
            nn.GELU(),
            nn.Dropout(0.15),
            nn.Linear(24, 1),
        )

    def forward(self, features):
        return self.network(features).squeeze(1)

fusion_mlp = FusionMLP(X_train.shape[1]).to(DEVICE)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(
    fusion_mlp.parameters(), lr=FUSION_LR, weight_decay=FUSION_WEIGHT_DECAY
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=2, min_lr=1e-6
)

@torch.inference_mode()
def mlp_probabilities(model, X, batch_size=2048):
    model.eval()
    output = []
    tensor = torch.as_tensor(X).float()
    for start in range(0, len(tensor), batch_size):
        logits = model(tensor[start:start+batch_size].to(DEVICE))
        output.extend(torch.sigmoid(logits).cpu().numpy().tolist())
    return np.asarray(output, dtype=np.float32)

In [ ]:
# ============================================================
# CELL 17 — TRAIN MLP WITH VALIDATION COLLAPSE PENALTY
# ============================================================
best_state = None
best_score = -np.inf
best_epoch = 0
stale = 0
history = []

for epoch in range(1, FUSION_EPOCHS+1):
    fusion_mlp.train()
    running_loss = 0.0
    count = 0

    for features, labels in train_loader:
        features = features.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)
        logits = fusion_mlp(features)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(fusion_mlp.parameters(), 5.0)
        optimizer.step()

        running_loss += loss.item() * len(labels)
        count += len(labels)

    val_probs = mlp_probabilities(fusion_mlp, X_val)
    threshold_row, _ = choose_threshold(
        y_val, val_probs, cache["val"]["scenarios"]
    )
    score = float(threshold_row["score"])
    scheduler.step(score)

    print(
        f"Epoch {epoch:02d} loss={running_loss/max(count,1):.4f} "
        f"score={score:.4f} macro_f1={threshold_row['macro_f1']:.4f} "
        f"real_rec={threshold_row['real_recall']:.4f} "
        f"fake_rec={threshold_row['fake_recall']:.4f} "
        f"worst_scenario={threshold_row['worst_scenario_recall']:.4f}"
    )

    history.append({"epoch": epoch, "loss": running_loss/max(count,1), **threshold_row.to_dict()})

    if score > best_score:
        best_score = score
        best_epoch = epoch
        best_state = copy.deepcopy(fusion_mlp.state_dict())
        stale = 0
    else:
        stale += 1
        if stale >= FUSION_PATIENCE:
            print("Early stopping.")
            break

fusion_mlp.load_state_dict(best_state)
mlp_val_probs = mlp_probabilities(fusion_mlp, X_val)
mlp_threshold_row, mlp_threshold_table = choose_threshold(
    y_val, mlp_val_probs, cache["val"]["scenarios"]
)
MLP_THRESHOLD = float(mlp_threshold_row["threshold"])

print("Best epoch:", best_epoch)
display(mlp_threshold_row.to_frame().T)

In [ ]:
# ============================================================
# CELL 18 — SELECT LOGISTIC OR MLP USING VALIDATION ONLY
# ============================================================
candidate_table = pd.DataFrame([
    {
        "model": "logistic",
        "score": float(log_threshold_row["score"]),
        "macro_f1": float(log_threshold_row["macro_f1"]),
        "worst_scenario": float(log_threshold_row["worst_scenario_recall"]),
        "threshold": LOGISTIC_THRESHOLD,
    },
    {
        "model": "mlp",
        "score": float(mlp_threshold_row["score"]),
        "macro_f1": float(mlp_threshold_row["macro_f1"]),
        "worst_scenario": float(mlp_threshold_row["worst_scenario_recall"]),
        "threshold": MLP_THRESHOLD,
    },
]).sort_values("score", ascending=False)

display(candidate_table)
SELECTED_MODEL = candidate_table.iloc[0]["model"]
SELECTED_THRESHOLD = float(candidate_table.iloc[0]["threshold"])
print("Selected fusion model:", SELECTED_MODEL)

In [ ]:
# ============================================================
# CELL 19 — EXPERT AND FUSION EVALUATION ON IDENTICAL SAMPLES
# ============================================================
def expert_probabilities(raw_X, expert):
    index = {"frequency": 0, "original_spatial": 1, "aux_spatial": 2}[expert]
    return 1.0 / (1.0 + np.exp(-raw_X[:, index]))

def selected_fusion_probs(split):
    X = transformed(split)
    if SELECTED_MODEL == "logistic":
        return logistic.predict_proba(X)[:, 1]
    return mlp_probabilities(fusion_mlp, X)

evaluation_rows = []

for split, title in (
    ("mixed_test", "MIXED HELD-OUT TEST"),
    ("ff_test", "PURE FF++ TEST"),
    ("celeb_test", "CELEBDF ZERO-SHOT"),
    ("phone_test", "PHONE HELD-OUT REAL"),
):
    data = cache[split]
    y = data["y"]

    print("\n\n", "#"*82, "\n", title)
    for expert in ("frequency", "original_spatial", "aux_spatial"):
        probs = expert_probabilities(data["X"], expert)
        # Expert thresholds are selected on fusion validation for fair same-domain comparison.
        val_expert_probs = expert_probabilities(cache["val"]["X"], expert)
        row, _ = choose_threshold(
            y_val, val_expert_probs, cache["val"]["scenarios"]
        )
        threshold = float(row["threshold"])
        metrics = print_metrics(
            f"{expert.upper()} — {title}", y, probs, threshold, data["scenarios"]
        )
        evaluation_rows.append({
            "split": split, "model": expert, "threshold": threshold,
            **{k: v for k, v in metrics.items() if k != "predictions"},
        })

    fusion_probs = selected_fusion_probs(split)
    metrics = print_metrics(
        f"SELECTED FUSION ({SELECTED_MODEL}) — {title}",
        y, fusion_probs, SELECTED_THRESHOLD, data["scenarios"]
    )
    evaluation_rows.append({
        "split": split, "model": f"fusion_{SELECTED_MODEL}",
        "threshold": SELECTED_THRESHOLD,
        **{k: v for k, v in metrics.items() if k != "predictions"},
    })

results_df = pd.DataFrame(evaluation_rows)
display(results_df)
results_df.to_csv(OUTPUT_DIR / "three_expert_fusion_results.csv", index=False)

In [ ]:
# ============================================================
# CELL 20 — ABLATION / PERMUTATION IMPORTANCE
# ============================================================
base_val_probs = selected_fusion_probs("val")
base_auc = roc_auc_score(y_val, base_val_probs)

importance_rows = []
rng = np.random.default_rng(SEED)

for index, name in enumerate(feature_names):
    raw = cache["val"]["X"].copy()
    raw[:, index] = rng.permutation(raw[:, index])
    shuffled = scaler.transform(raw)

    if SELECTED_MODEL == "logistic":
        probs = logistic.predict_proba(shuffled)[:, 1]
    else:
        probs = mlp_probabilities(fusion_mlp, shuffled)

    importance_rows.append({
        "feature": name,
        "auc_drop": base_auc - roc_auc_score(y_val, probs),
    })

importance_df = pd.DataFrame(importance_rows).sort_values("auc_drop", ascending=False)
display(importance_df)
importance_df.to_csv(OUTPUT_DIR / "fusion_permutation_importance.csv", index=False)

In [ ]:
# ============================================================
# CELL 21 — SAVE COMPLETE PACKAGE
# ============================================================
package = {
    "selected_model": SELECTED_MODEL,
    "selected_threshold": SELECTED_THRESHOLD,
    "feature_names": feature_names,
    "scaler_mean": scaler.mean_,
    "scaler_scale": scaler.scale_,
    "logistic_coef": logistic.coef_,
    "logistic_intercept": logistic.intercept_,
    "mlp_state_dict": fusion_mlp.state_dict(),
    "mlp_input_dim": X_train.shape[1],
    "validation_candidates": candidate_table.to_dict("records"),
    "paths": {
        "frequency_checkpoint": str(FREQ_CKPT),
        "original_spatial_checkpoint": str(ORIGINAL_SPATIAL_CKPT),
        "auxiliary_head_checkpoint": str(AUXILIARY_HEAD_CKPT),
    },
}
torch.save(package, OUTPUT_DIR / "best_three_expert_fusion.pth")

pd.DataFrame(history).to_csv(OUTPUT_DIR / "fusion_training_history.csv", index=False)
log_threshold_table.to_csv(OUTPUT_DIR / "logistic_threshold_sweep.csv", index=False)
mlp_threshold_table.to_csv(OUTPUT_DIR / "mlp_threshold_sweep.csv", index=False)

print("Saved:")
for filename in (
    "best_three_expert_fusion.pth",
    "three_expert_fusion_results.csv",
    "fusion_permutation_importance.csv",
    "fusion_training_history.csv",
):
    print(" -", OUTPUT_DIR / filename)

In [ ]:
# ============================================================
# CELL 22 — FINAL ACCEPTANCE GATES
# ============================================================
fusion_name = f"fusion_{SELECTED_MODEL}"
fusion_results = results_df[results_df["model"] == fusion_name].set_index("split")

gates = {
    "FF++ macro-F1 >= 0.90":
        fusion_results.loc["ff_test", "macro_f1"] >= 0.90,
    "FF++ real recall >= 0.88":
        fusion_results.loc["ff_test", "real_recall"] >= 0.88,
    "FF++ fake recall >= 0.92":
        fusion_results.loc["ff_test", "fake_recall"] >= 0.92,
    "CelebDF AUC >= 0.76":
        fusion_results.loc["celeb_test", "auc"] >= 0.76,
    "CelebDF real recall >= 0.70":
        fusion_results.loc["celeb_test", "real_recall"] >= 0.70,
    "CelebDF fake recall >= 0.70":
        fusion_results.loc["celeb_test", "fake_recall"] >= 0.70,
    "Phone real recall >= 0.80":
        fusion_results.loc["phone_test", "real_recall"] >= 0.80,
}

for name, passed in gates.items():
    print(("PASS" if passed else "FAIL"), "-", name)

print(f"\nPassed {sum(gates.values())}/{len(gates)} acceptance gates.")
print(
    "Keep the fusion model only if it beats or complements the experts "
    "without a class/domain collapse."
)